# 06 — Figure Export for Report

Exports all publication-quality PNG figures for the LaTeX report.
All figures use the **corrected** per-window methodology.

In [ ]:
# Setup

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed
from src.nig import NIGParams, nig_pdf, nig_cdf
from src.assessment import pit_qq, pit_ks_test, anderson_darling_pit

FIGURE_DIR = Path("../report/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Export settings
EXPORT_WIDTH  = 1200
EXPORT_HEIGHT = 500
EXPORT_SCALE  = 2

def save_fig(fig, name, width=EXPORT_WIDTH, height=EXPORT_HEIGHT):
    """Save figure as PNG for LaTeX report."""
    path = FIGURE_DIR / f"{name}.png"
    fig.write_image(str(path), width=width, height=height, scale=EXPORT_SCALE)
    print(f"  Saved: {path}")

print("Setup OK")

In [ ]:
# Load all data

innovations  = load_processed("innovations.parquet")
sigma_fore   = load_processed("sigma_forecasts.parquet")
var99        = load_processed("var99.parquet")
var95        = load_processed("var95.parquet")
pit_nig_df   = load_processed("pit_nig.parquet")
pit_t_df     = load_processed("pit_t.parquet")
pit_gauss_df = load_processed("pit_gauss.parquet")
master_df    = load_processed("master_assessment.parquet").set_index("Ticker")
exc_df       = load_processed("exceedance_results.parquet")

# Full returns for log-return plots
returns_full = load_processed("returns.parquet")

TICKERS = list(innovations.columns)
colors  = dict(zip(TICKERS, px.colors.qualitative.Plotly))

# Load per-asset rolling results
all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

print(f"Loaded {len(TICKERS)} assets")

## Figure 1 — Return Distributions vs Gaussian

In [ ]:
fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.04)

x_grid = np.linspace(-7, 7, 400)
gauss_ref = stats.norm.pdf(x_grid)

for i, ticker in enumerate(TICKERS):
    innov = innovations[ticker].dropna().values
    fig.add_trace(go.Histogram(
        x=innov, nbinsx=60, histnorm="probability density",
        marker_color=colors[ticker], opacity=0.5, showlegend=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=gauss_ref, mode="lines",
        line=dict(color="black", width=1.5),
        showlegend=(i == 0), name="Gaussian N(0,1)",
    ), row=1, col=i+1)
    fig.update_xaxes(range=[-5, 5], row=1, col=i+1)

fig.update_layout(
    title="Standardised Innovation Distributions vs Gaussian",
    height=350, template="plotly_white",
)
save_fig(fig, "fig1_return_distributions", height=350)
fig.show()

## Figure 2 — NIG vs Student-T vs Gaussian PDF Fits

In [ ]:
x_grid = np.linspace(-7, 7, 500)
gauss_ref = stats.norm.pdf(x_grid)

fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.04)

for i, ticker in enumerate(TICKERS):
    innov = innovations[ticker].dropna().values
    df_r  = all_results[ticker]

    # Median NIG params (representative)
    nig_rep = NIGParams(
        alpha=df_r["nig_alpha"].median(), beta=df_r["nig_beta"].median(),
        mu=df_r["nig_mu"].median(), delta=df_r["nig_delta"].median(),
    )
    nig_y = nig_pdf(x_grid, nig_rep)

    # Median Student-T dof (representative)
    t_dof = df_r["t_dof"].median()
    t_y = stats.t.pdf(x_grid, t_dof, loc=0, scale=1)

    fig.add_trace(go.Histogram(
        x=innov, nbinsx=60, histnorm="probability density",
        marker_color="lightgrey", opacity=0.5, showlegend=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=nig_y, mode="lines",
        line=dict(color="#d62728", width=2),
        showlegend=(i == 0), name="NIG",
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=t_y, mode="lines",
        line=dict(color="#ff7f0e", width=1.5, dash="dash"),
        showlegend=(i == 0), name="Student-T",
    ), row=1, col=i+1)
    fig.add_trace(go.Scatter(
        x=x_grid, y=gauss_ref, mode="lines",
        line=dict(color="black", width=1, dash="dot"),
        showlegend=(i == 0), name="Gaussian",
    ), row=1, col=i+1)
    fig.update_xaxes(range=[-5, 5], row=1, col=i+1)

fig.update_layout(
    title="Distribution Fits: NIG vs Student-T vs Gaussian (median params)",
    height=350, template="plotly_white",
)
save_fig(fig, "fig2_distribution_fits", height=350)
fig.show()

## Figure 3 — VaR vs Actual Returns

In [ ]:
# All assets combined panel
fig = make_subplots(
    rows=len(TICKERS), cols=1, shared_xaxes=True,
    subplot_titles=TICKERS, vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    actual = all_results[ticker]["return"]
    v99, v95 = var99[ticker], var95[ticker]
    exceed = actual < v99

    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values, mode="lines",
        line=dict(color=colors[ticker], width=0.7), showlegend=False,
    ), row=i+1, col=1)
    fig.add_trace(go.Scatter(
        x=v99.index, y=v99.values, mode="lines",
        name="VaR 99%" if i == 0 else None, showlegend=(i == 0),
        line=dict(color="#e63946", width=1.5),
    ), row=i+1, col=1)
    fig.add_trace(go.Scatter(
        x=v95.index, y=v95.values, mode="lines",
        name="VaR 95%" if i == 0 else None, showlegend=(i == 0),
        line=dict(color="#c47000", width=1.2, dash="dot"),
    ), row=i+1, col=1)
    fig.add_trace(go.Scatter(
        x=actual.index[exceed], y=actual.values[exceed], mode="markers",
        name="Exceedance" if i == 0 else None, showlegend=(i == 0),
        marker=dict(color="#040720", size=5, symbol="x"),
    ), row=i+1, col=1)
    fig.update_yaxes(title_text="Log return", row=i+1, col=1)

fig.update_layout(
    title="Actual Returns vs VaR — All Assets",
    height=1000, template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", y=-0.04),
)
save_fig(fig, "fig3_var_all_assets", height=1000)
fig.show()

In [ ]:
# Individual per-asset VaR figures
for ticker in TICKERS:
    actual = all_results[ticker]["return"]
    v99, v95 = var99[ticker], var95[ticker]
    exceed = actual < v99

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values, mode="lines",
        name="Returns", line=dict(color=colors[ticker], width=0.8),
    ))
    fig.add_trace(go.Scatter(
        x=v99.index, y=v99.values, mode="lines",
        name="VaR 99%", line=dict(color="#e63946", width=1.5),
    ))
    fig.add_trace(go.Scatter(
        x=v95.index, y=v95.values, mode="lines",
        name="VaR 95%", line=dict(color="#c47000", width=1.2, dash="dot"),
    ))
    fig.add_trace(go.Scatter(
        x=actual.index[exceed], y=actual.values[exceed], mode="markers",
        name="Exceedance", marker=dict(color="#040720", size=6, symbol="x"),
    ))
    fig.update_layout(
        title=f"{ticker} — Returns vs VaR",
        height=400, template="plotly_white",
    )
    safe = ticker.replace("^", "").replace("=", "_")
    save_fig(fig, f"fig3_var_{safe}", height=400)

print("Individual VaR figures saved")

## Figure 4 — VaR Backtesting Table

In [ ]:
color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}
font_map  = {"GREEN": "#144a23", "RED": "#5c0a10", "BLUE": "#0d2b40"}

exc = exc_df.copy()

fig = go.Figure(data=[go.Table(
    columnwidth=[90, 60, 80, 80, 70, 80],
    header=dict(
        values=["Ticker", "VaR level", "Expected",
                "Exceedances", "p-value", "Result"],
        fill_color="#2c3e50",
        font=dict(color="white", size=12),
        align="center", height=32,
    ),
    cells=dict(
        values=[exc[c].tolist() for c in ["Ticker", "VaR level", "Expected",
                                           "Exceedances", "p-value", "Result"]],
        fill_color=[
            ["#f8f9fa"] * len(exc)] * 5 + [
            [color_map.get(r, "#f8f9fa") for r in exc["Result"]],
        ],
        font=dict(
            color=[["#2c3e50"] * len(exc)] * 5 + [
                [font_map.get(r, "#2c3e50") for r in exc["Result"]],
            ],
            size=12,
        ),
        align="center", height=28,
    ),
)])

fig.update_layout(
    title="VaR Backtesting Results — All Levels",
    template="plotly_white", height=400,
    margin=dict(t=50, b=10, l=10, r=10),
)
save_fig(fig, "fig4_var_backtest_table", height=400)
fig.show()

## Figure 5 — Master Assessment Table

In [ ]:
def result_color(val):
    return {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}.get(val, "#f8f9fa")

m = master_df.reset_index()

col_headers = [
    "Ticker", "VaR99<br>Exceed", "VaR99<br>p-val", "VaR99<br>Result",
    "VaR95<br>Exceed", "VaR95<br>p-val", "VaR95<br>Result",
    "KS<br>(NIG)", "KS p", "AD<br>(NIG)",
    "Christ<br>99% p", "Christ<br>95% p",
]

col_keys = ["Ticker", "VaR99 exceed", "VaR99 p-val", "VaR99 result",
            "VaR95 exceed", "VaR95 p-val", "VaR95 result",
            "KS (NIG)", "KS p", "AD (NIG)",
            "Christ 99% p", "Christ 95% p"]

cell_values = [m[k].tolist() for k in col_keys]
n_rows = len(m)
default_fill = ["#f8f9fa"] * n_rows
fill_colors = [default_fill[:] for _ in col_keys]
fill_colors[3] = [result_color(r) for r in m["VaR99 result"]]
fill_colors[6] = [result_color(r) for r in m["VaR95 result"]]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=col_headers, fill_color="#2c3e50",
        font=dict(color="white", size=11), align="center", height=36,
    ),
    cells=dict(
        values=cell_values, fill_color=fill_colors,
        font=dict(size=11), align="center", height=28,
    ),
)])

fig.update_layout(
    title="Master Assessment Table", template="plotly_white",
    height=320, margin=dict(t=50, b=10, l=10, r=10),
)
save_fig(fig, "fig5_master_table", height=320)
fig.show()

## Figure 6 — PIT QQ Plots (NIG vs Student-T vs Gaussian)

In [ ]:
pit_dfs = {"NIG": pit_nig_df, "T": pit_t_df, "Gauss": pit_gauss_df}
pit_colors = {"NIG": "#d62728", "T": "#ff7f0e", "Gauss": "#1f77b4"}

fig = make_subplots(
    rows=len(TICKERS), cols=3,
    column_titles=["NIG PIT QQ", "Student-T PIT QQ", "Gaussian PIT QQ"],
    row_titles=TICKERS,
    vertical_spacing=0.04, horizontal_spacing=0.05,
)

for i, ticker in enumerate(TICKERS):
    for j, (label, pdf) in enumerate(pit_dfs.items()):
        pit = pdf[ticker].dropna().values
        emp_q, mod_q = pit_qq(pit)

        fig.add_trace(go.Scatter(
            x=emp_q, y=mod_q, mode="markers",
            marker=dict(color=pit_colors[label], size=2),
            showlegend=False,
        ), row=i+1, col=j+1)
        fig.add_trace(go.Scatter(
            x=[0,1], y=[0,1], mode="lines", showlegend=False,
            line=dict(color="black", width=0.8, dash="dash"),
        ), row=i+1, col=j+1)
        fig.update_xaxes(
            title_text="Empirical quantile",
            title_font=dict(size=9, color="rgba(0,0,0,0.4)"),
            title_standoff=2, range=[0,1], row=i+1, col=j+1,
        )
        fig.update_yaxes(
            title_text="Model quantile",
            title_font=dict(size=9, color="rgba(0,0,0,0.4)"),
            title_standoff=2, range=[0,1], row=i+1, col=j+1,
        )

fig.update_layout(
    title="PIT QQ Plots Against Uniform(0,1)",
    height=200 * len(TICKERS), template="plotly_white",
)
save_fig(fig, "fig6_pit_qq", height=200 * len(TICKERS))
fig.show()

## Figure 7 — Log Return Time Series

In [ ]:
fig = make_subplots(
    rows=3, cols=2, subplot_titles=TICKERS + [""],
    vertical_spacing=0.08, horizontal_spacing=0.08,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1)]
for idx, ticker in enumerate(TICKERS):
    row, col = positions[idx]
    r = returns_full[ticker]
    fig.add_trace(go.Scatter(
        x=r.index, y=r.values, mode="lines",
        line=dict(color=colors[ticker], width=0.6), showlegend=False,
    ), row=row, col=col)
    fig.add_hline(y=0, line_dash="dash", line_color="grey",
                  line_width=0.5, row=row, col=col)
    fig.update_yaxes(title_text="Log return", row=row, col=col)

fig.update_layout(
    title="Daily Log Returns — Full History (2005–2025)",
    height=700, template="plotly_white",
)
save_fig(fig, "fig7_log_returns", height=700)
fig.show()

## Figure 8 — Conditional Volatility

In [ ]:
fig = make_subplots(
    rows=len(TICKERS), cols=1, shared_xaxes=True,
    subplot_titles=TICKERS, vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    sigma = sigma_fore[ticker] * np.sqrt(252)  # annualised
    fig.add_trace(go.Scatter(
        x=sigma.index, y=sigma.values, mode="lines",
        name=ticker, line=dict(color=colors[ticker], width=0.8),
    ), row=i+1, col=1)
    fig.update_yaxes(title_text="Ann. vol", row=i+1, col=1)

fig.update_layout(
    title="GARCH(1,1) Conditional Volatility (Annualised)",
    height=900, template="plotly_white", hovermode="x unified",
)
save_fig(fig, "fig8_conditional_volatility", height=900)
fig.show()

## Figure 9 — PIT Histograms

In [ ]:
fig = make_subplots(
    rows=2, cols=3, subplot_titles=TICKERS + [""],
    vertical_spacing=0.12, horizontal_spacing=0.06,
)

for idx, ticker in enumerate(TICKERS):
    row, col = idx // 3 + 1, idx % 3 + 1
    pit = pit_nig_df[ticker].dropna().values
    fig.add_trace(go.Histogram(
        x=pit, nbinsx=20, histnorm="probability density",
        marker_color=colors[ticker], opacity=0.7, showlegend=False,
    ), row=row, col=col)
    fig.add_hline(y=1.0, line_dash="dash", line_color="red",
                  line_width=1.5, row=row, col=col)
    fig.update_xaxes(range=[0, 1], row=row, col=col)

fig.update_layout(
    title="NIG PIT Histograms — Should Be Flat If Model Correct",
    height=500, template="plotly_white",
)
save_fig(fig, "fig9_pit_histograms", height=500)
fig.show()

---

In [ ]:
# Verify all figures exported

expected = [
    "fig1_return_distributions.png",
    "fig2_distribution_fits.png",
    "fig3_var_all_assets.png",
    "fig4_var_backtest_table.png",
    "fig5_master_table.png",
    "fig6_pit_qq.png",
    "fig7_log_returns.png",
    "fig8_conditional_volatility.png",
    "fig9_pit_histograms.png",
]

# Per-asset VaR figures
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    expected.append(f"fig3_var_{safe}.png")

missing = [f for f in expected if not (FIGURE_DIR / f).exists()]

if missing:
    print(f"WARNING: {len(missing)} figures missing:")
    for f in missing:
        print(f"  {f}")
else:
    print(f"All {len(expected)} figures exported successfully.")
    print(f"Output directory: {FIGURE_DIR.resolve()}")

print(f"\nFigure list:")
for f in sorted(FIGURE_DIR.glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:7.1f} KB")

---